# InspecSafe 自动标注 Colab 首版\n\n目标：在 Google Colab 上快速跑通 `GroundingDINO + SAM` 的 RGB 自动标注流程。\n

In [ ]:
!nvidia-smi\n!pip install -q opencv-python pillow pycocotools supervision\n%cd /content\n!test -d /content/GroundingDINO || git clone https://github.com/IDEA-Research/GroundingDINO.git\n!test -d /content/segment-anything || git clone https://github.com/facebookresearch/segment-anything.git\n!pip install -q -e /content/GroundingDINO\n!pip install -q -e /content/segment-anything\n!mkdir -p /content/models\n!wget -q -O /content/models/groundingdino_swint_ogc.pth https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth\n!wget -q -O /content/models/sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
from google.colab import drive\nfrom pathlib import Path\n\ndrive.mount('/content/drive', force_remount=False)\nDATASET_ROOT = Path('/content/drive/MyDrive/InspecSafe-V1/DATA_PATH')\nOUTPUT_ROOT = Path('/content/drive/MyDrive/InspecSafe-V1/auto_label_outputs')\nif not DATASET_ROOT.exists():\n    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\nprint(f'DATASET_ROOT={DATASET_ROOT}')\nprint(f'OUTPUT_ROOT={OUTPUT_ROOT}')\n\nPROMPTS = [\n    'person', 'open flame', 'fire hydrant', 'fire extinguisher',\n    'safety helmet', 'electrical box', 'electronic control cabinet',\n    'pipeline', 'valve', 'pressure gauge'\n]\nBOX_THRESHOLD = 0.30\nTEXT_THRESHOLD = 0.25\nMAX_IMAGES = 20

In [ ]:
import json\nimport cv2\nimport numpy as np\nimport torch\n\nfrom groundingdino.util.inference import Model\nfrom segment_anything import sam_model_registry, SamPredictor\n\ndef collect_rgb_images(dataset_root, max_images=None):\n    image_paths = []\n    for split in ('train', 'test'):\n        split_root = dataset_root / split / 'Annotations'\n        if split_root.exists():\n            for pattern in ('*.jpg', '*.jpeg', '*.png'):\n                image_paths.extend(sorted(split_root.rglob(pattern)))\n    image_paths = sorted(set(image_paths))\n    return image_paths[:max_images] if max_images else image_paths\n\ndef mask_to_polygon(mask):\n    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    if not contours:\n        return []\n    contour = max(contours, key=cv2.contourArea).squeeze(axis=1)\n    if contour.ndim != 2 or len(contour) < 3:\n        return []\n    return contour.astype(float).tolist()\n\nDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\ngrounding_model = Model(\n    model_config_path='/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',\n    model_checkpoint_path='/content/models/groundingdino_swint_ogc.pth',\n)\nsam = sam_model_registry['vit_b'](checkpoint='/content/models/sam_vit_b_01ec64.pth')\nsam.to(device=DEVICE)\nsam_predictor = SamPredictor(sam)\nimages = collect_rgb_images(DATASET_ROOT, MAX_IMAGES)\nif not images:\n    raise RuntimeError(f'No RGB images found under {DATASET_ROOT}')\nprint(f'Collected {len(images)} images, device={DEVICE}')\nlen(images)

In [ ]:
def run_single_image(image_path):\n    image_bgr = cv2.imread(str(image_path))\n    if image_bgr is None:\n        raise ValueError(f'Failed to read image: {image_path}')\n    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n    detections = grounding_model.predict_with_classes(\n        image=image_rgb,\n        classes=PROMPTS,\n        box_threshold=BOX_THRESHOLD,\n        text_threshold=TEXT_THRESHOLD,\n    )\n    sam_predictor.set_image(image_rgb)\n    preds = []\n    for box, score, class_id in zip(detections.xyxy, detections.confidence, detections.class_id):\n        transformed_box = sam_predictor.transform.apply_boxes(np.array([box], dtype=np.float32), image_rgb.shape[:2])\n        masks, _, _ = sam_predictor.predict_torch(\n            point_coords=None,\n            point_labels=None,\n            boxes=torch.tensor(transformed_box, dtype=torch.float32, device=DEVICE),\n            multimask_output=False,\n        )\n        polygon = mask_to_polygon(masks[0, 0].detach().cpu().numpy().astype(np.uint8))\n        if not polygon:\n            x1, y1, x2, y2 = [float(v) for v in box.tolist()]\n            polygon = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]\n        preds.append({\n            'label': PROMPTS[int(class_id)],\n            'score': float(score),\n            'bbox': [float(v) for v in box.tolist()],\n            'polygon': polygon,\n        })\n    return {\n        'image_path': str(image_path),\n        'width': int(image_rgb.shape[1]),\n        'height': int(image_rgb.shape[0]),\n        'predictions': preds,\n    }\n\nrecords = []\nfor i, image_path in enumerate(images, start=1):\n    try:\n        record = run_single_image(image_path)\n        records.append(record)\n        print(f'[{i}/{len(images)}] {image_path.name}: {len(record["predictions"])} predictions')\n    except Exception as exc:\n        print(f'[{i}/{len(images)}] failed: {image_path.name}: {exc}')

In [ ]:
def polygon_area(points):\n    if len(points) < 3:\n        return 0.0\n    area = 0.0\n    for idx, point in enumerate(points):\n        nxt = points[(idx + 1) % len(points)]\n        area += point[0] * nxt[1] - nxt[0] * point[1]\n    return abs(area) / 2.0\n\ndef polygon_to_bbox(points):\n    xs = [p[0] for p in points]\n    ys = [p[1] for p in points]\n    return [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]\n\ndef flatten_polygon(points):\n    flat = []\n    for x_coord, y_coord in points:\n        flat.extend([x_coord, y_coord])\n    return flat\n\ndef to_coco(records):\n    categories = {}\n    images_payload = []\n    annotations_payload = []\n    ann_id = 1\n    for image_id, record in enumerate(records, start=1):\n        images_payload.append({\n            'id': image_id,\n            'file_name': record['image_path'],\n            'width': record['width'],\n            'height': record['height'],\n        })\n        for pred in record['predictions']:\n            categories.setdefault(pred['label'], len(categories) + 1)\n            annotations_payload.append({\n                'id': ann_id,\n                'image_id': image_id,\n                'category_id': categories[pred['label']],\n                'bbox': polygon_to_bbox(pred['polygon']),\n                'segmentation': [flatten_polygon(pred['polygon'])],\n                'area': polygon_area(pred['polygon']),\n                'iscrowd': 0,\n                'score': pred.get('score'),\n            })\n            ann_id += 1\n    categories_payload = [\n        {'id': cid, 'name': name} for name, cid in sorted(categories.items(), key=lambda item: item[1])\n    ]\n    return {'images': images_payload, 'annotations': annotations_payload, 'categories': categories_payload}\n\nwith (OUTPUT_ROOT / 'predictions.jsonl').open('w', encoding='utf-8') as handle:\n    for record in records:\n        handle.write(json.dumps(record, ensure_ascii=False) + '\\n')\ncoco_payload = to_coco(records)\n(OUTPUT_ROOT / 'predictions_coco.json').write_text(json.dumps(coco_payload, indent=2, ensure_ascii=False), encoding='utf-8')\nprint(OUTPUT_ROOT / 'predictions.jsonl')\nprint(OUTPUT_ROOT / 'predictions_coco.json')